# Embedding

In [36]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [37]:
q1 = 'Can I still join the course after the start date?'
v1 = model.encode(q1)
v1.shape

(384,)

In [38]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [39]:
# compare the query against the document using dot product
v1.dot(dv)

np.float32(0.3233239)

In [40]:
q2 = 'How to install Docker on Windows?'
v2 = model.encode(q2)

In [41]:
v2.dot(dv)

np.float32(0.019730505)

The first score for `q1` vs `d` (0.32) is higher, so that query is more similar to the document about registration. 
The second score for `q2` vs `d` sits near 0, because installing Docker has nothing to do with registration. 
A score near 0 means the two vectors are about as different as they can be.

# Embedding Dataset

In [42]:
from ingest import load_faq_data

documents = load_faq_data()

In [43]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

Generating embeddings

Each document is a Python dictionary with a question and an answer. We
embed both together. That way a query can match against the question
text and the answer text in our index.

Build one text per document

In [44]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [45]:
len(texts)

1350

Now we generate the embeddings. We have about 1350 texts here. We won't
hand the model all of them at once. That takes a long time, and we can't
see what's happening inside. Instead we split them into batches.

First we import `tqdm` so we can watch the progress.
Next we chunk the dataset into batches of 50 and encode each batch.

In [46]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

We turn them into a 2-dimensional array (matrix) where

- rows are documents (vectors)
- columns are dimensions of the vectors

In [47]:
import numpy as np
X = np.array(vectors)
X.shape

(1350, 384)

Calling `X.shape` returns (1350, 384) - number of documents vs number of dimensions.

# Vector Search

In [48]:
# We have a matrix `X` with all document embeddings. We take a query,
# compare it against every document, and keep the most similar ones.

# When a query comes in, we embed it:

query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [56]:
# Next, we compute the dot product against all documents:

scores = X.dot(v_query)

In [ ]:
# This is matrix-vector multiplication. Each element `i` of `scores` is 
# the cosine similarity between document `i` (row `i` of `X`) and
# `v_query`.

# We could compute the same thing with a for loop:

# scores = [v_query.dot(X[i]) for i in range(len(X))]


In [57]:
# The highest score is the most similar document:
idx = np.argmax(scores)
idx, scores[idx]


(np.int64(2), np.float32(0.7629409))

In [58]:
# Let's see which document it is:

documents[idx]



{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [59]:
# `np.argsort` sorts from lowest to highest, so the last 5 are the top ones

top5 = np.argsort(scores)[-5:]
top5 

array([  7, 538, 907, 625,   2])

In [60]:
# They come out smallest-first, so we reverse them to get the highest first

top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [61]:
scores[top5]

array([0.7629409 , 0.757937  , 0.7192131 , 0.6536312 , 0.56009996],
      dtype=float32)

In [ ]:
# There's a shorter trick. We can negate the scores first, so the largest becomes the smallest. 
# Then `argsort` puts the best matches at the front.
top5 = np.argsort(-scores)[:5]

In [63]:
top5

array([  2, 625, 907, 538,   7])

In [64]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629409
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.757937
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

# Vector Search with minsearch

In [ ]:
# We pass the numpy array X with all embeddings and the list of documents as payload. 
# The keyword_fields parameter works the same as in the text Index, so we can filter by course later.

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [ ]:
# Let's search for a question:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(
            query_vector, 
            num_results=5
)
# Under the hood, it computes the dot product between each vector (after filtering) and our query vector.

In [67]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [68]:
# Like the text index, we can filter by keyword fields.
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

# RAG with Vector Search

In [69]:
from dotenv import load_dotenv
from groq import Groq
import os
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
# Load and index the data
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [71]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=client,
)

In [72]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)


'Yes—you can still join. If you’d like to earn a certificate, just be sure to submit your project before the submission deadline ends.'

This still uses keyword search. Text search isn't bad here, so the
answer may already look right. Next we replace search with vector
search.
We already have:

- All the indexed documents `documents`
- The embeddings matrix `X` with all these documents
- The vector search engine `vindex`

We can't pass `vindex` to RAG as-is. Text search takes the query string
directly, but vector search needs the query as a vector first. So we 
subclass `RAGBase` and override `search` to encode the query before
searching.

In [ ]:
# The `__init__` method adds one extra argument, `embedder`, for the
# sentence transformer. Inside `search` we use it to turn the query into a
# vector. Then we query `vindex` with that vector instead of the raw text.
# Everything else is inherited from `RAGBase`.

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [74]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=client,
)



In [75]:
query = 'I just found out about the program, can I still sign up?'
vector_assistant.rag(query)


'Yes—you can still join. Just keep in mind that if you’d like to earn a certificate, you’ll need to submit your project before the submission deadline.'

# Vector Search with sqlitesearch

In [76]:
# Creating the index
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)


In [ ]:
# Fit the index with our vectors and documents
vs_index.fit(vectors, documents)
# The index is saved to `faq_vectors2.db`. Unlike minsearch, this file persists on disk.


In [78]:
# Encode, then search
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [79]:
# Filtering by course
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [80]:
vs_index.close()